In [3]:
import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from datetime import datetime

# Machine Learning
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTE

# Scikit-Learn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

print("✅ All libraries imported successfully.")

✅ All libraries imported successfully.


In [4]:
PROJECT_ROOT = Path.cwd().parent

DATASET_DIR = PROJECT_ROOT / "dataset"

train_df = pd.read_csv(DATASET_DIR / "transactions_train.csv")

print("Dataset Shape :", train_df.shape)

train_df.head()

Dataset Shape : (300113, 21)


,transaction_id,transaction_time,customer_id,merchant_id,account_age_days,credit_score_band,kyc_level,avg_monthly_spend,merchant_risk_score,transaction_amount,...,device_type,is_international,ip_risk_score,txn_count_1h,txn_count_24h,failed_txn_count_24h,geo_distance_from_last_txn,amount_deviation_from_user_mean,is_fraud,post_auth_risk_score
0,359131,2023-01-01 00:02:00.328105993,11102,2282,284,2,3,6091.747132,0.456269,2408.320473,...,desktop,0,0.142532,1,3,1,33.458018,2205.262235,0,0.099920
1,351207,2023-01-01 00:02:26.339769237,22891,3016,1363,2,3,3794.044563,0.449021,2765.255095,...,mobile,0,0.131811,0,5,0,3.375083,2638.786943,0,0.291715
2,10209,2023-01-01 00:06:54.145825305,3102,1855,1318,5,2,6697.058451,0.220252,1529.079168,...,desktop,0,0.322137,0,5,0,13.732603,1305.843886,0,0.216647
3,62660,2023-01-01 00:06:57.723185583,4041,2525,1914,1,1,2906.711704,0.202223,610.407487,...,mobile,0,0.171764,1,2,0,18.840187,513.517097,0,0.354154
4,384254,2023-01-01 00:08:05.487541188,3979,1555,360,2,3,5082.651983,0.171230,986.397163,...,mobile,0,0.248766,1,1,0,15.344375,816.975430,0,0.149084


In [5]:
print("=" * 50)
print("Dataset Information")
print("=" * 50)

train_df.info()

print("\nMissing Values\n")
print(train_df.isnull().sum())

print("\nFraud Distribution\n")
print(train_df["is_fraud"].value_counts())

Dataset Information
<class 'pandas.DataFrame'>
RangeIndex: 300113 entries, 0 to 300112
Data columns (total 21 columns):
 #   Column                           Non-Null Count   Dtype  
---  ------                           --------------   -----  
 0   transaction_id                   300113 non-null  int64  
 1   transaction_time                 300113 non-null  str    
 2   customer_id                      300113 non-null  int64  
 3   merchant_id                      300113 non-null  int64  
 4   account_age_days                 300113 non-null  int64  
 5   credit_score_band                300113 non-null  int64  
 6   kyc_level                        300113 non-null  int64  
 7   avg_monthly_spend                300113 non-null  float64
 8   merchant_risk_score              300113 non-null  float64
 9   transaction_amount               300113 non-null  float64
 10  payment_channel                  300113 non-null  str    
 11  device_type                      300113 non-null  str   

In [6]:
# Convert timestamp into features

train_df["transaction_time"] = pd.to_datetime(train_df["transaction_time"])

train_df["hour"] = train_df["transaction_time"].dt.hour
train_df["day_of_week"] = train_df["transaction_time"].dt.dayofweek
train_df["month"] = train_df["transaction_time"].dt.month

# Drop original timestamp
train_df.drop(columns=["transaction_time"], inplace=True)

print(train_df.shape)
train_df.head()

(300113, 23)


,transaction_id,customer_id,merchant_id,account_age_days,credit_score_band,kyc_level,avg_monthly_spend,merchant_risk_score,transaction_amount,payment_channel,...,txn_count_1h,txn_count_24h,failed_txn_count_24h,geo_distance_from_last_txn,amount_deviation_from_user_mean,is_fraud,post_auth_risk_score,hour,day_of_week,month
0,359131,11102,2282,284,2,3,6091.747132,0.456269,2408.320473,wallet,...,1,3,1,33.458018,2205.262235,0,0.099920,0,6,1
1,351207,22891,3016,1363,2,3,3794.044563,0.449021,2765.255095,bank_transfer,...,0,5,0,3.375083,2638.786943,0,0.291715,0,6,1
2,10209,3102,1855,1318,5,2,6697.058451,0.220252,1529.079168,card,...,0,5,0,13.732603,1305.843886,0,0.216647,0,6,1
3,62660,4041,2525,1914,1,1,2906.711704,0.202223,610.407487,card,...,1,2,0,18.840187,513.517097,0,0.354154,0,6,1
4,384254,3979,1555,360,2,3,5082.651983,0.171230,986.397163,card,...,1,1,0,15.344375,816.975430,0,0.149084,0,6,1


In [11]:
TARGET = "is_fraud"

DROP_COLUMNS = [
    "transaction_id",
    "customer_id",
    "merchant_id",
    "post_auth_risk_score",
]

X = train_df.drop(columns=DROP_COLUMNS + [TARGET])

y = train_df[TARGET]

print(X.shape)
print(y.shape)

(300113, 18)
(300113,)


In [9]:
print(train_df.columns.tolist())

['transaction_id', 'customer_id', 'merchant_id', 'account_age_days', 'credit_score_band', 'kyc_level', 'avg_monthly_spend', 'merchant_risk_score', 'transaction_amount', 'payment_channel', 'device_type', 'is_international', 'ip_risk_score', 'txn_count_1h', 'txn_count_24h', 'failed_txn_count_24h', 'geo_distance_from_last_txn', 'amount_deviation_from_user_mean', 'is_fraud', 'post_auth_risk_score', 'hour', 'day_of_week', 'month']


In [10]:
for i, col in enumerate(train_df.columns):
    print(f"{i+1:2d}. {col}")

 1. transaction_id
 2. customer_id
 3. merchant_id
 4. account_age_days
 5. credit_score_band
 6. kyc_level
 7. avg_monthly_spend
 8. merchant_risk_score
 9. transaction_amount
10. payment_channel
11. device_type
12. is_international
13. ip_risk_score
14. txn_count_1h
15. txn_count_24h
16. failed_txn_count_24h
17. geo_distance_from_last_txn
18. amount_deviation_from_user_mean
19. is_fraud
20. post_auth_risk_score
21. hour
22. day_of_week
23. month


In [12]:
categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()

numerical_columns = [
    col for col in X.columns
    if col not in categorical_columns
]

print("Categorical Columns:")
print(categorical_columns)

print()

print("Number of Numerical Columns:", len(numerical_columns))
print(numerical_columns)

Categorical Columns:
['payment_channel', 'device_type']

Number of Numerical Columns: 16
['account_age_days', 'credit_score_band', 'kyc_level', 'avg_monthly_spend', 'merchant_risk_score', 'transaction_amount', 'is_international', 'ip_risk_score', 'txn_count_1h', 'txn_count_24h', 'failed_txn_count_24h', 'geo_distance_from_last_txn', 'amount_deviation_from_user_mean', 'hour', 'day_of_week', 'month']


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training :", X_train.shape)
print("Testing  :", X_test.shape)

Training : (240090, 18)
Testing  : (60023, 18)


In [14]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OrdinalEncoder(
                handle_unknown="use_encoded_value",
                unknown_value=-1
            ),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print(X_train_encoded.shape)
print(X_test_encoded.shape)

(240090, 18)
(60023, 18)


In [15]:
smote = SMOTE(random_state=42)

X_train_balanced, y_train_balanced = smote.fit_resample(
    X_train_encoded,
    y_train
)

print(pd.Series(y_train_balanced).value_counts())

is_fraud
0    236193
1    236193
Name: count, dtype: int64


In [16]:
model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=10,
    min_child_weight=3,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_balanced,
    y_train_balanced
)

print("✅ Model Training Complete")

[LightGBM] [Info] Number of positive: 236193, number of negative: 236193
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.004230 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4590
[LightGBM] [Info] Number of data points in the train set: 472386, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
✅ Model Training Complete


In [17]:
probabilities = model.predict_proba(X_test_encoded)[:, 1]

predictions = (probabilities >= 0.32).astype(int)

print("Predictions Generated")

Predictions Generated


In [18]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

print("=" * 50)

print(f"Accuracy : {accuracy_score(y_test, predictions):.4f}")

print(f"Precision: {precision_score(y_test, predictions):.4f}")

print(f"Recall   : {recall_score(y_test, predictions):.4f}")

print(f"F1 Score : {f1_score(y_test, predictions):.4f}")

print(f"ROC AUC  : {roc_auc_score(y_test, probabilities):.4f}")

print("=" * 50)

Accuracy : 0.9841
Precision: 0.5171
Recall   : 0.2957
F1 Score : 0.3762
ROC AUC  : 0.8152


In [21]:
from sklearn.pipeline import Pipeline

fraud_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMClassifier(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=10,
        min_child_weight=3,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        n_jobs=-1,
    ))
])

fraud_pipeline.fit(X_train, y_train)

print("✅ Production pipeline trained successfully.")

[LightGBM] [Info] Number of positive: 3897, number of negative: 236193
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005650 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1876
[LightGBM] [Info] Number of data points in the train set: 240090, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.016231 -> initscore=-4.104442
[LightGBM] [Info] Start training from score -4.104442
✅ Production pipeline trained successfully.


In [22]:
import json
import joblib
from pathlib import Path
from datetime import datetime

MODEL_DIR = PROJECT_ROOT / "backend" / "app" / "ml"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(
    fraud_pipeline,
    MODEL_DIR / "fraud_pipeline.pkl"
)

joblib.dump(
    threshold,
    MODEL_DIR / "threshold.pkl"
)

metadata = {
    "model": "LightGBM Pipeline",
    "version": "1.0.0",
    "accuracy": float(accuracy_score(y_test, predictions)),
    "precision": float(precision_score(y_test, predictions)),
    "recall": float(recall_score(y_test, predictions)),
    "f1_score": float(f1_score(y_test, predictions)),
    "roc_auc": float(roc_auc_score(y_test, probabilities)),
    "threshold": threshold,
    "trained_on": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
}

with open(MODEL_DIR / "metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("✅ Pipeline saved.")
print("📁", MODEL_DIR)

✅ Pipeline saved.
📁 /Users/nakulsingh/Documents/AI-Fraud Detection/backend/app/ml
